# Singapore Smart City: Gold Layer (Feature Store & PyG Graphs)

This notebook represents the **Gold Layer** of our Medallion Data Engineering Architecture ($0 Cloud-Native Execution).

### Pipeline Objectives:
1. **Load Silver Data:** Read the fast, compressed `traffic_features.parquet` from Google Drive.
2. **Spatial Topology (Adjacency Matrix):** Calculate the Haversine distance between all 90 LTA cameras to build the Graph Adjacency Matrix ($A$).
3. **PyTorch Geometric (PyG) Datasets:** Convert the tabular Parquet data into ST-GNN ready `Data` objects containing Node Features ($X$), Edge Indices ($E$), and Edge Weights ($W$).
4. **Export Gold Data:** Save the serialized PyTorch tensors back to Google Drive so the `PI-STGNN` Level 3 model can instantly load them for training.

> **Cost:** $0, runs freely on Colab CPUs.

In [ ]:
!pip install -q pandas pyarrow torch torch-geometric haversine tqdm

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from haversine import haversine, Unit
from torch_geometric.data import Data
from tqdm.auto import tqdm

DRIVE_ROOT = Path('/content/drive/MyDrive/sg_smart_city')
SILVER_DIR = DRIVE_ROOT / 'data' / 'silver'
GOLD_DIR = DRIVE_ROOT / 'data' / 'gold'

os.makedirs(GOLD_DIR, exist_ok=True)
print(f"Target Gold Feature Store: {GOLD_DIR}")

In [ ]:
# 1. Load Silver Parquet
parquet_path = SILVER_DIR / 'traffic_features.parquet'
if not parquet_path.exists():
    raise FileNotFoundError(f"{parquet_path} not found. Run the Silver ETL notebook first.")

print("Loading Silver Parquet data...")
df = pd.read_parquet(parquet_path)
print(f"Loaded {len(df)} records across {df['camera_id'].nunique()} cameras.")

In [ ]:
# 2. Build Spatial Topology (Adjacency Matrix)
print("\n--- Building Spatial Adjacency Matrix ---")
camera_metadata_path = DRIVE_ROOT / 'data' / 'raw' / 'metadata'  # Assumes a master camera list exists

# Fallback: Extract unique cameras and their coordinates dynamically from the Silver data
# (Assuming the original API returned lat/lon inside the metadata.jsonl)

camera_coords = {}
for _, row in df.drop_duplicates(subset=['camera_id']).iterrows():
    # Standard LTA API format usually includes 'latitude' and 'longitude' in the 'location' dict
    if 'location' in row and isinstance(row['location'], dict):
        lat = row['location'].get('latitude', 0.0)
        lon = row['location'].get('longitude', 0.0)
        camera_coords[row['camera_id']] = (lat, lon)
    else:
        # Placeholder if location was not parsed flat in Silver yet
        camera_coords[row['camera_id']] = (1.3521, 103.8198) # Default SG center

cameras = list(camera_coords.keys())
camera_to_idx = {cam: idx for idx, cam in enumerate(cameras)}
num_nodes = len(cameras)

edge_indices = []
edge_weights = []

# Create edges based on distance threshold (e.g. connected if within 3km)
DISTANCE_THRESHOLD_KM = 3.0

for i in tqdm(range(num_nodes), desc="Calculating Haversine distances"):
    for j in range(num_nodes):
        if i != j:
            dist = haversine(camera_coords[cameras[i]], camera_coords[cameras[j]], unit=Unit.KILOMETERS)
            if dist <= DISTANCE_THRESHOLD_KM:
                edge_indices.append([i, j])
                # Convert distance to weight (closer = higher weight)
                weight = np.exp(-(dist**2))
                edge_weights.append(weight)

edge_index_tensor = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()
edge_weight_tensor = torch.tensor(edge_weights, dtype=torch.float)

print(f"\nCreated Spatial Graph:\n- Nodes: {num_nodes}\n- Edges: {edge_index_tensor.shape[1]}\n- Average Degree: {edge_index_tensor.shape[1] / num_nodes:.2f}")

In [ ]:
# 3. Export to PyTorch Tensors (Gold Format)

# Save the Graph Topology
torch.save({
    'edge_index': edge_index_tensor,
    'edge_weight': edge_weight_tensor,
    'camera_to_idx': camera_to_idx
}, GOLD_DIR / 'spatial_graph_topology.pt')

print(f"\n✅ Success! Gold Layer PyG Topology saved to {GOLD_DIR / 'spatial_graph_topology.pt'}")

print("\n>> NEXT STEP: The YOLO Level 1 Model will append Node Features (vehicle counts) to this graph during the Training phase.")